In [9]:
import pandas as pd
import json

## Data loading

In [19]:
# load counts
data_FIB = pd.read_csv("../../Moment-equations/Real-Data-2/Data/GSE151334_FIB_counts_thresh.csv", index_col=0)

# load RNA types
biotypes_dict = json.load(open("../../Moment-equations/Real-Data-2/Biotypes/biotypes_FIB.json"))

# select indices of protein coding mRNA and non-coding miRNA
pcRNA_indices = [idx for idx, btype in enumerate(biotypes_dict.values()) if btype == "protein_coding"]
miRNA_indices = [idx for idx, btype in enumerate(biotypes_dict.values()) if btype == "miRNA"]

# select protein coding genes
data_FIB_pcRNA_NB = data_FIB.iloc[pcRNA_indices]

# select miRNA
data_FIB_miRNA_NB = data_FIB.iloc[miRNA_indices]

In [20]:
# load BD results
BD_df = pd.read_csv("./Results/BD_concat.csv", index_col=0)

## Running

In [21]:
# mRNA
#G = 100

# miRNA
miRNA_list_NB = data_FIB_miRNA_NB.index

# phases
phases_NB = ["All"]#, "G1", "S", "G2M"]

In [22]:
# select miRNA
miRNA_NB = "MIR199A1"
print(f"miRNA {miRNA_NB}")

# settings
d_NB = 3

# for each phase
for phase_NB in phases_NB:

    # select BD infeasible mRNA
    mRNA_NB = BD_df.index[BD_df[f"{miRNA_NB}_{phase_NB}_d3_c95_status"] == "INFEASIBLE"]
    mRNA_string_NB = " ".join(list(mRNA_NB))

    # run
    outfile_NB = f"./Results/Script-Outfiles/TE_{miRNA_NB}_{phase_NB}.csv"
    %run -i telegraph_script.py --miRNA {miRNA_NB} --mRNA {mRNA_string_NB} --phase {phase_NB} --d {d_NB} --outfile {outfile_NB}

miRNA MIR199A1


100%|██████████| 96/96 [01:12<00:00,  1.32it/s]


In [ ]:
# load result files
df_list = []
for miRNA_NB in data_FIB_miRNA_NB.index:
    for phase_NB in phases_NB:
        try:
            df = pd.read_csv(f"./Results/Script-Outfiles/TE_{miRNA_NB}_{phase_NB}.csv", index_col=0)
            df_list.append(df)
        except FileNotFoundError:
            pass

# concat
df_concat = pd.concat(df_list, axis="columns")

# save
#df_concat.to_csv("./Results/TE_concat.csv")

In [24]:
df_concat.value_counts()

MIR199A1_All_d3_c95_status
OPTIMAL                       78
CUT_LIMIT                     18
Name: count, dtype: int64